# 07 - Closed-orbit correction

This example measures the BPM-by-corrector response numerically and applies an SVD pseudoinverse. The same pattern works with a measured response matrix.

In [ ]:
import Pkg
EXAMPLES_DIR = isfile(joinpath(pwd(), "common.jl")) ? pwd() : joinpath(pwd(), "examples")
Pkg.activate(EXAMPLES_DIR)
using TrackPad, StaticArrays
include(joinpath(EXAMPLES_DIR, "common.jl"))
using .TrackPadExamples

using LinearAlgebra

In [ ]:
_, beam = madx_fodo()
function bpm_orbit(kicks)
    ring = instrumented_fodo(corrector_kicks=kicks)
    closed = find_closed_orbit_4d(ring, beam)
    initial = @SVector [closed[1], closed[2], closed[3], closed[4], 0.0, 0.0]
    orbit = boundary_orbit(ring, beam, initial)
    bpm_values(ring, orbit[:, 1])
end

uncorrected = bpm_orbit(zeros(4))

In [ ]:
h = 1e-7
response = hcat(((bpm_orbit(h .* [j == i for j in 1:4]) -
                   bpm_orbit(-h .* [j == i for j in 1:4])) / (2h)
                 for i in 1:4)...)
correction = -pinv(response; rtol=1e-10) * uncorrected
corrected = bpm_orbit(correction)

rms(v) = norm(v) / sqrt(length(v))
(rms_before=1e6 * rms(uncorrected), rms_after=1e6 * rms(corrected),
 corrector_kicks_urad=1e6 .* correction)

The response columns are corrector kicks in radians and rows are BPM positions in metres. Inspect singular values and set the pseudoinverse cutoff from measurement noise and corrector limits.